# 00 — Data Loading

**Purpose:** Crawl all ShuttleSet match folders, concatenate every `set{n}.csv` into one master dataframe, parse player names from folder names, and save outputs for downstream notebooks.

**Outputs:**
- `EDA/data/shuttleset_master.parquet` — all strokes, all matches
- `EDA/data/player_map.csv` — parsed A/B player names per match

In [17]:
import pandas as pd
from pathlib import Path
import importlib
import config
importlib.reload(config)
from config import PLAYERS, TOURNAMENT_KEYWORDS, FOLDER_TYPOS

In [18]:
NOTEBOOK_DIR = Path("__file__").resolve().parent
PROJECT_ROOT = NOTEBOOK_DIR.parent

SHUTTLESET_PATH = PROJECT_ROOT / "data" / "CoachAI-Projects" / "ShuttleSet" / "set"
OUTPUT_DIR = NOTEBOOK_DIR / "data"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Project root:    {PROJECT_ROOT}")
print(f"ShuttleSet path: {SHUTTLESET_PATH}")
print(f"Output dir:      {OUTPUT_DIR}")
print(f"Path exists:     {SHUTTLESET_PATH.exists()}")

assert SHUTTLESET_PATH.exists(), f"Path not found: {SHUTTLESET_PATH}"

# Build player lookup
PLAYER_TOKEN_MAP = {}
for player in PLAYERS:
    key = player.lower().replace(" ", "_")
    PLAYER_TOKEN_MAP[key] = player
    if "-" in player:
        alt_key = player.lower().replace(" ", "_")   # hyphen stays
        PLAYER_TOKEN_MAP[alt_key] = player

Project root:    /Users/nidhipad/Dropbox/Mac/Downloads/Cognitive-Modeling-Capstone-Project
ShuttleSet path: /Users/nidhipad/Dropbox/Mac/Downloads/Cognitive-Modeling-Capstone-Project/data/CoachAI-Projects/ShuttleSet/set
Output dir:      /Users/nidhipad/Dropbox/Mac/Downloads/Cognitive-Modeling-Capstone-Project/EDA/data
Path exists:     True


## 1. Crawl all match folders and concatenate CSVs

In [19]:
def load_all_csvs() -> pd.DataFrame:
    records = []
    skipped = []

    for match_dir in sorted(SHUTTLESET_PATH.iterdir()):
        if not match_dir.is_dir():
            continue
        csv_files = sorted(match_dir.glob("set*.csv"))
        if not csv_files:
            skipped.append(match_dir.name)
            continue
        for csv_file in csv_files:
            set_num = int(csv_file.stem.replace("set", ""))
            df = pd.read_csv(csv_file)
            df["match_id"] = match_dir.name
            df["set_num"] = set_num
            records.append(df)

    master = pd.concat(records, ignore_index=True)
    master["stroke_id"] = range(len(master))

    print(f"Loaded: {len(master):,} strokes")
    print(f"Matches: {master['match_id'].nunique()}")
    print(f"Sets:    {master.groupby(['match_id','set_num']).ngroups}")
    print(f"Rallies: {master.groupby(['match_id','set_num','rally']).ngroups}")
    if skipped:
        print(f"\nSkipped (no CSVs found): {skipped}")

    return master

## 2. Parse player names from folder names

Folder names follow the pattern:
`PlayerA_PlayerB_TOURNAMENT_YEAR_Round`

Match player A and player B to correct player names and split on tournament keywords

In [20]:
def split_player_names(match_id: str) -> tuple[str, str]:
    normalized = match_id.strip().replace(" ", "_")
    parts = normalized.split("_")

    split_idx = None
    for i, p in enumerate(parts):
        if p.strip().upper() in TOURNAMENT_KEYWORDS or (
            len(p.strip()) == 4 and p.strip().isdigit()
        ):
            split_idx = i
            break

    player_tokens = parts[:split_idx] if split_idx is not None else parts
    player_str = "_".join(t.strip() for t in player_tokens if t.strip()).lower()

    name_a, name_b = None, None
    for token_key, canonical_name in PLAYER_TOKEN_MAP.items():
        if player_str.startswith(token_key):
            name_a = canonical_name
            remainder = player_str[len(token_key):].lstrip("_")
            # Check exact match first, then typo map
            name_b = (
                PLAYER_TOKEN_MAP.get(remainder)
                or FOLDER_TYPOS.get(remainder)
            )
            if name_b:
                break

    if not name_a or not name_b:
        print(f"Could not parse: {match_id}")
        print(f"player_str='{player_str}'")
        mid = len(player_tokens) // 2
        name_a = name_a or " ".join(player_tokens[:mid])
        name_b = name_b or " ".join(player_tokens[mid:])

    return name_a, name_b

In [21]:
def build_player_map(master: pd.DataFrame) -> pd.DataFrame:
    map_rows = []
    for match_id in master["match_id"].unique():
        name_a, name_b = split_player_names(match_id)
        map_rows.append({"match_id": match_id, "A": name_a, "B": name_b})

    player_map = pd.DataFrame(map_rows).sort_values("match_id").reset_index(drop=True)

    print(f"Player map rows: {len(player_map)}")
    player_map.head(10)

    return player_map

## 3. Save Outputs to files in EDA/data
- `EDA/data/shuttleset_master.parquet` — all strokes, all matches
- `EDA/data/player_map.csv` — parsed A/B player names per match

In [22]:
def save_outputs(master: pd.DataFrame, player_map: pd.DataFrame):
    master_path = OUTPUT_DIR / "shuttleset_master.parquet"
    map_path    = OUTPUT_DIR / "player_map.csv"

    master.to_parquet(master_path, index=False)
    player_map.to_csv(map_path, index=False)

    print(f"\nSaved: {master_path}")
    print(f"Saved: {map_path}")

In [25]:
def main():
    master_pd=load_all_csvs()

    player_map_pd=build_player_map(master_pd)

    pd.set_option("display.max_colwidth", 80)
    pd.set_option("display.max_rows", 200)
    print(player_map_pd.to_string(index=False))

    save_outputs(master_pd, player_map_pd)



In [26]:
if __name__ == "__main__":
    main()

Loaded: 36,484 strokes
Matches: 44
Sets:    104
Rallies: 3683
Player map rows: 44
                                                                             match_id                                A                                B
             An_Se_Young_Pornpawee_Chochuwong_TOYOTA_THAILAND_OPEN_2021_QuarterFinals                      An Se Young             Pornpawee Chochuwong
                 An_Se_Young_Ratchanok_Intanon_YONEX_Thailand_Open_2021_QuarterFinals                      An Se Young                Ratchanok Intanon
                Anders_ANTONSEN_Jonatan_CHRISTIE Indonesia_Masters_2020_QuarterFinals                  Anders Antonsen                 Jonatan Christie
                 Anders_Antonsen_Sameer_Verma_TOYOTA_THAILAND_OPEN_2021_QuarterFinals                  Anders Antonsen                     Sameer Verma
                Anders_Antonsen_Viktor_Axelsen_HSBC_BWF_WORLD_TOUR_FINALS_2020_Finals                  Anders Antonsen                   Viktor Axelsen
      